# Community Proposal Board — Blockchain Mini Project
**Student:** Võ Khắc Tuyên · **Student ID:** 30219042961  
**Course:** DS 441 A — Blockchain: Theory & Practices

This notebook connects to a Solidity smart contract on local Ganache using web3.py. Its saved outputs show real transactions, receipts, events, rejected actions, and the final contract state.

The first code cell follows the supplied connection template: `RPC_URL`, `CONTRACT_ADDRESS`, and a literal `ABI_JSON` array. The address and ABI belong to CommunityProposalBoard. No external configuration file is required to run this notebook.

To rerun: start Ganache, deploy the Solidity contract, update `CONTRACT_ADDRESS` and `ABI_JSON` if necessary, select a Python kernel with web3.py installed, and choose **Restart Kernel and Run All Cells**. Each run creates a new proposal. Use test accounts and test Ether only. GitHub can display the saved outputs without running Ganache.

## 1. Connect using the supplied template
Set the Ganache RPC URL, deployed contract address, and complete ABI JSON array directly below. The template validates the connection, address, ABI format, and deployed bytecode, then displays the Chain ID, contract address, and available unlocked accounts. The ABI is for CommunityProposalBoard rather than the sample store/retrieve contract.

In [1]:
import json
from web3 import Web3
from web3.utils.abi import abi_to_signature

RPC_URL = 'http://127.0.0.1:7545'
CONTRACT_ADDRESS = '0x1661710adE37BEEBb806aAA6643E4cd281d6F165'  # Paste the deployed contract address here.
ABI_JSON = '''[
  {
    "anonymous": false,
    "inputs": [
      {
        "indexed": true,
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      },
      {
        "indexed": true,
        "internalType": "address",
        "name": "author",
        "type": "address"
      }
    ],
    "name": "ProposalClosed",
    "type": "event"
  },
  {
    "anonymous": false,
    "inputs": [
      {
        "indexed": true,
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      },
      {
        "indexed": true,
        "internalType": "address",
        "name": "author",
        "type": "address"
      },
      {
        "indexed": false,
        "internalType": "string",
        "name": "title",
        "type": "string"
      }
    ],
    "name": "ProposalCreated",
    "type": "event"
  },
  {
    "anonymous": false,
    "inputs": [
      {
        "indexed": true,
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      },
      {
        "indexed": true,
        "internalType": "address",
        "name": "supporter",
        "type": "address"
      },
      {
        "indexed": false,
        "internalType": "uint256",
        "name": "endorsementCount",
        "type": "uint256"
      }
    ],
    "name": "ProposalEndorsed",
    "type": "event"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      }
    ],
    "name": "closeProposal",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "string",
        "name": "title",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "description",
        "type": "string"
      }
    ],
    "name": "createProposal",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "id",
        "type": "uint256"
      }
    ],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      }
    ],
    "name": "endorseProposal",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "proposalId",
        "type": "uint256"
      }
    ],
    "name": "getProposal",
    "outputs": [
      {
        "components": [
          {
            "internalType": "uint256",
            "name": "id",
            "type": "uint256"
          },
          {
            "internalType": "address",
            "name": "author",
            "type": "address"
          },
          {
            "internalType": "string",
            "name": "title",
            "type": "string"
          },
          {
            "internalType": "string",
            "name": "description",
            "type": "string"
          },
          {
            "internalType": "uint256",
            "name": "endorsementCount",
            "type": "uint256"
          },
          {
            "internalType": "bool",
            "name": "closed",
            "type": "bool"
          }
        ],
        "internalType": "struct CommunityProposalBoard.Proposal",
        "name": "",
        "type": "tuple"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      },
      {
        "internalType": "address",
        "name": "",
        "type": "address"
      }
    ],
    "name": "hasEndorsed",
    "outputs": [
      {
        "internalType": "bool",
        "name": "",
        "type": "bool"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "proposalCount",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  }
]'''

w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'timeout': 10}))
if not w3.is_connected():
    raise ConnectionError(f'Cannot connect to Ganache at {RPC_URL}. Check that Ganache is running and the URL is correct.')
if not Web3.is_address(CONTRACT_ADDRESS):
    raise ValueError('Paste a valid deployed contract address into CONTRACT_ADDRESS.')
try:
    abi = json.loads(ABI_JSON)
except json.JSONDecodeError as exc:
    raise ValueError('Paste the complete ABI JSON array into ABI_JSON.') from exc
if not isinstance(abi, list):
    raise ValueError('ABI_JSON must contain a JSON array.')

address = Web3.to_checksum_address(CONTRACT_ADDRESS)
if not w3.eth.get_code(address):
    raise ValueError('No contract code at this address on the connected network. Check the address and Ganache network.')
contract = w3.eth.contract(address=address, abi=abi)
accounts = w3.eth.accounts

print('Connected to Ganache and contract!')
print('Chain ID:', w3.eth.chain_id)
print('Contract:', address)
print('Available unlocked accounts:', accounts)

Connected to Ganache and contract!
Chain ID: 1337
Contract: 0x1661710adE37BEEBb806aAA6643E4cd281d6F165
Available unlocked accounts: ['0x195Cab71A742e0Bb60a756cDa133767Cf95E1073', '0xA2C5a72040C6951aD1B83224685457802f33b528', '0x7C1cfe07cbdDc532061840591F58642EF6ebfE9D', '0xbbFf9B778C8A5c1EFCdeF458db31a29B588F9397', '0xEe8BBc437cF8eaEe45B7A1679DdC74D3a791f79F', '0xF60C47e9f55faA8D6eda1b9cA374a86BE399dA5d', '0xa67e5036b1451140D96836AC7Fd906883b92B345', '0x134911Fd8D4A67D37bbB681d9b1085ee3c88575C', '0xec68eBD9d8801C4e369BEAF96dcd19cf86731a21', '0x830c8d2F318831f2B7d2514C59D80FA12BBe48b0']


## 2. Demo accounts and helper functions
Use the first three Ganache test accounts as A (author), B (supporter), and C (newcomer). Print the function signatures using the template's abi_to_signature import.

`send` submits a real transaction and checks its successful receipt. `reject` verifies the revert reason using `eth_call`, then submits a real transaction with a fixed gas limit and requires a receipt with **status = 0**. Reverted transactions are still included in blocks and consume test gas, while their state changes and events are rolled back.

In [2]:
from web3.exceptions import Web3Exception
from urllib.parse import urlparse

if urlparse(RPC_URL).hostname not in {'localhost', '127.0.0.1', '::1'}:
    raise ValueError('Use local Ganache only.')
if not any(name in w3.client_version.lower() for name in ('ganache', 'testrpc')):
    raise ValueError('Expected a Ganache test blockchain.')
if len(accounts) < 3:
    raise ValueError('The demonstration needs at least three unlocked Ganache test accounts.')
A, B, C = accounts[:3]
results = []
print('A - author:', A)
print('B - supporter:', B)
print('C - newcomer:', C)
print('Contract functions:')
for entry in abi:
    if entry.get('type') == 'function':
        print(' -', abi_to_signature(entry))

def passed(label):
    results.append((label, "PASS"))
    print("PASS:", label)

def send(label, function, sender, event_name=None):
    tx_hash = function.transact({"from": sender})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    print("Transaction:", Web3.to_hex(tx_hash))
    print("Block:", receipt.blockNumber, "| status:", receipt.status, "| gas:", receipt.gasUsed)
    assert receipt.status == 1, f"Transaction failed: {label}"
    if event_name:
        logs = getattr(contract.events, event_name)().process_receipt(receipt)
        assert len(logs) == 1, f"Expected event {event_name}"
        print("Event:", event_name, dict(logs[0]["args"]))
    passed(label)
    return receipt

def proposal_data(proposal_id):
    fields = ("id", "author", "title", "description", "endorsementCount", "closed")
    return dict(zip(fields, contract.functions.getProposal(proposal_id).call()))

def show(proposal_id):
    print(json.dumps(proposal_data(proposal_id), ensure_ascii=False, indent=2))

def reject(label, function, sender, expected_reason):
    try:
        function.call({"from": sender})
    except (Web3Exception, ValueError) as exc:
        message = str(exc)
        assert expected_reason in message and "revert" in message.lower(), message
        print("Expected reason:", expected_reason)
        rpc_error = (getattr(exc, "rpc_response", None) or {}).get("error", {})
        print("Contract error:", rpc_error.get("message", message))
    else:
        raise AssertionError("Invalid action was accepted: " + label)
    try:
        tx_hash = function.transact({"from": sender, "gas": 250000})
    except Web3Exception as exc:
        # Ganache GUI may return a JSON-RPC error even after mining the revert.
        error = (getattr(exc, "rpc_response", None) or {}).get("error", {})
        data = error.get("data", {})
        if not isinstance(data, dict) or not data.get("hash"):
            raise
        assert expected_reason in str(exc), str(exc)
        tx_hash = Web3.to_bytes(hexstr=data["hash"])
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    print("Rejected transaction:", Web3.to_hex(tx_hash))
    print("Block:", receipt.blockNumber, "| status:", receipt.status, "| gas:", receipt.gasUsed)
    assert receipt.status == 0, "Invalid transaction must revert"
    assert len(receipt.logs) == 0, "Reverted transaction must not retain events"
    passed(label)

A - author: 0x195Cab71A742e0Bb60a756cDa133767Cf95E1073
B - supporter: 0xA2C5a72040C6951aD1B83224685457802f33b528
C - newcomer: 0x7C1cfe07cbdDc532061840591F58642EF6ebfE9D
Contract functions:
 - closeProposal(uint256)
 - createProposal(string,string)
 - endorseProposal(uint256)
 - getProposal(uint256)
 - hasEndorsed(uint256,address)
 - proposalCount()


## 3. Account A creates a proposal
Read the proposal ID from the ProposalCreated event so the notebook can be rerun on a blockchain that already contains proposals. Verify the author, the initial endorsement count of zero, and the open status.

In [3]:
receipt = send(
    "A creates a proposal",
    contract.functions.createProposal(
        "Trồng cây tại công viên",
        "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng."
    ),
    A,
    "ProposalCreated"
)
proposal_id = contract.events.ProposalCreated().process_receipt(receipt)[0]["args"]["proposalId"]
initial = proposal_data(proposal_id)
assert initial["author"] == A and initial["endorsementCount"] == 0 and initial["closed"] is False
print("New proposal ID:", proposal_id)
show(proposal_id)

Transaction:

 0x3dd8b4aa43176246670a96f3f0405842a9afdd2c8bae0b690fe37f6b2dadc7aa
Block: 52 | status: 1 | gas: 193014
Event: ProposalCreated {'proposalId': 5, 'author': '0x195Cab71A742e0Bb60a756cDa133767Cf95E1073', 'title': 'Trồng cây tại công viên'}
PASS: A creates a proposal
New proposal ID: 5
{
  "id": 5,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 0,
  "closed": false
}


## 4. Reject self-endorsement
The contract must reject A's attempt to endorse their own proposal. The endorsement count remains zero, and the mapping must not record an endorsement.

In [4]:
reject("Self endorsement rejected", contract.functions.endorseProposal(proposal_id), A, "Author cannot endorse own proposal")
assert proposal_data(proposal_id)["endorsementCount"] == 0
assert not contract.functions.hasEndorsed(proposal_id, A).call()

Expected reason: Author cannot endorse own proposal
Contract error: VM Exception while processing transaction: revert Author cannot endorse own proposal
Rejected transaction: 0x355e2fe0a103c76dc4719213e56bb7955f7ae67c7fffd96c409a991f334e9795
Block: 53 | status: 0 | gas: 28073
PASS: Self endorsement rejected


## 5. Account B endorses successfully
The count increases to one, the mapping records B's endorsement, and the contract emits ProposalEndorsed.

In [5]:
send("B endorses the proposal", contract.functions.endorseProposal(proposal_id), B, "ProposalEndorsed")
assert proposal_data(proposal_id)["endorsementCount"] == 1
assert contract.functions.hasEndorsed(proposal_id, B).call()
show(proposal_id)

Transaction: 0xd4cb1bc9857fecb4f02d343a86548889bf966e474d72d2bc47d01839441accf5
Block: 54 | status: 1 | gas: 74612
Event: ProposalEndorsed {'proposalId': 5, 'supporter': '0xA2C5a72040C6951aD1B83224685457802f33b528', 'endorsementCount': 1}
PASS: B endorses the proposal
{
  "id": 5,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": false
}


## 6. Reject a duplicate endorsement
B's second endorsement must revert with Already endorsed. The count remains one.

In [6]:
reject("Duplicate endorsement rejected", contract.functions.endorseProposal(proposal_id), B, "Already endorsed")
assert proposal_data(proposal_id)["endorsementCount"] == 1

Expected reason: Already endorsed
Contract error: VM Exception while processing transaction: revert Already endorsed
Rejected transaction: 0x555edd595a7dae074f1933a710600629911c384caee3b32fe604db27c9febb21
Block: 55 | status: 0 | gas: 30318
PASS: Duplicate endorsement rejected


## 7. Reject an unauthorized closure
B is not the author, so the contract rejects the closure attempt. The proposal remains open.

In [7]:
reject("Unauthorized close rejected", contract.functions.closeProposal(proposal_id), B, "Only author can close proposal")
assert proposal_data(proposal_id)["closed"] is False

Expected reason: Only author can close proposal
Contract error: VM Exception while processing transaction: revert Only author can close proposal
Rejected transaction: 0xf4237ad0fcbf56c00a00079fdfe4051ebcc193cba6973b8a0a4bb532790a88d9
Block: 56 | status: 0 | gas: 25860
PASS: Unauthorized close rejected


## 8. Account A closes the proposal
The transaction succeeds, emits ProposalClosed, and changes closed to true.

In [8]:
send("A closes the proposal", contract.functions.closeProposal(proposal_id), A, "ProposalClosed")
assert proposal_data(proposal_id)["closed"] is True
show(proposal_id)

Transaction: 0x9aebf506a0493975661cbf3cb67b5936102522504b4dcb877526d42db6414790
Block: 57 | status: 1 | gas: 49550
Event: ProposalClosed {'proposalId': 5, 'author': '0x195Cab71A742e0Bb60a756cDa133767Cf95E1073'}
PASS: A closes the proposal


{
  "id": 5,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": true
}


## 9. Reject endorsements after closure
Use C, who has not endorsed the proposal, to demonstrate that the rejection is caused by the closed state. Also verify that the author cannot close the proposal a second time.

In [9]:
assert not contract.functions.hasEndorsed(proposal_id, C).call()
reject("Endorsement after closure rejected", contract.functions.endorseProposal(proposal_id), C, "Proposal is closed")
assert not contract.functions.hasEndorsed(proposal_id, C).call()
reject("Repeated closure rejected", contract.functions.closeProposal(proposal_id), A, "Proposal is closed")

Expected reason: Proposal is closed
Contract error: VM Exception while processing transaction: revert Proposal is closed
Rejected transaction: 0x06c9596ba39a140f24a0f54a2342b77846bcd807a2749034439e96c0acdb378d
Block: 58 | status: 0 | gas: 25909
PASS: Endorsement after closure rejected
Expected reason: Proposal is closed
Contract error: VM Exception while processing transaction: revert Proposal is closed


Rejected transaction: 0x0efaa047b7ab4b46afa032d6260ebf2705bfc0779239a1d06a00345b92edd4c8
Block: 59 | status: 0 | gas: 27998
PASS: Repeated closure rejected


## 10. Reject invalid inputs
The contract must reject a nonexistent proposal ID and an empty title. A failed proposal-creation attempt must not increase the total proposal count.

In [10]:
invalid_id = contract.functions.proposalCount().call() + 1000
reject("Unknown proposal rejected", contract.functions.endorseProposal(invalid_id), C, "Proposal does not exist")
count_before = contract.functions.proposalCount().call()
reject("Empty title rejected", contract.functions.createProposal("", "Description"), A, "Title must be 1-120 bytes")
assert contract.functions.proposalCount().call() == count_before

Expected reason: Proposal does not exist
Contract error: VM Exception while processing transaction: revert Proposal does not exist


Rejected transaction: 0x15370c9ba96ef59735951a56808253903252c49880952ec6d541e7db10cd6ea8
Block: 60 | status: 0 | gas: 23735
PASS: Unknown proposal rejected


Expected reason: Title must be 1-120 bytes
Contract error: VM Exception while processing transaction: revert Title must be 1-120 bytes
Rejected transaction: 0xacab42c258c81b667f800c20bdc5b91392afd3216fbb3b8ebccf954c714b78f0
Block: 61 | status: 0 | gas: 22700
PASS: Empty title rejected


## 11. Results and final state
The proposal belongs to A, has exactly one endorsement from B, and is closed. C has not endorsed it. Each PASS entry is added only after its corresponding transaction checks succeed. Any failed assertion stops notebook execution.

In [11]:
final = proposal_data(proposal_id)
assert final["author"] == A
assert final["endorsementCount"] == 1
assert final["closed"] is True
assert contract.functions.hasEndorsed(proposal_id, B).call()
assert not contract.functions.hasEndorsed(proposal_id, C).call()
passed("Final state and endorsement mappings are correct")
show(proposal_id)
print("=" * 72)
for index, (label, outcome) in enumerate(results, 1):
    print(f"{index:02}. {outcome:4} | {label}")
print("=" * 72)
assert len(results) == 11
print(f"RESULT: {len(results)}/{len(results)} checks passed.")
print("All transaction hashes, receipts and events above were obtained from Ganache.")

PASS:

 Final state and endorsement mappings are correct


{
  "id": 5,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": true
}
01. PASS | A creates a proposal
02. PASS | Self endorsement rejected
03. PASS | B endorses the proposal
04. PASS | Duplicate endorsement rejected
05. PASS | Unauthorized close rejected
06. PASS | A closes the proposal
07. PASS | Endorsement after closure rejected
08. PASS | Repeated closure rejected
09. PASS | Unknown proposal rejected
10. PASS | Empty title rejected
11. PASS | Final state and endorsement mappings are correct
RESULT: 11/11 checks passed.
All transaction hashes, receipts and events above were obtained from Ganache.


## Explanation and references
- Solidity enforces permissions through msg.sender, independently of the interface.
- The hasEndorsed mapping allows one endorsement per address for each proposal.
- Only the author may close a proposal; closed proposals reject further endorsements.
- Outputs were obtained from local Ganache. They contain no private keys or real funds.
- [web3.py](https://web3py.readthedocs.io/en/stable/overview.html), [Solidity](https://docs.soliditylang.org/), [Ganache](https://github.com/trufflesuite/ganache).
- The code, notebook, and documentation were developed with assistance from OpenAI Codex. The student should understand the implementation and be able to explain it before submission.